# Хеш-таблицы


**Хеш-таблица** (hash map) — структура, которая хранит пары `ключ → значение` с доступом в среднем за `O(1)`.

В Python это встроенные `dict` и `set`. Большинство задач на собеседовании решаются через них.


## Как это работает

1. **Хеш-функция** превращает ключ в число: `hash(key) → int`.
2. От этого числа берут остаток по размеру таблицы: `index = hash(key) % capacity`.
3. Значение кладут в **бакет** по этому индексу.

Если два разных ключа дают один индекс — это **коллизия**. Решают двумя способами:
- **Цепочки**: в каждом бакете хранится связный список пар.
- **Открытая адресация**: ищется следующий свободный слот (linear/quadratic probing).

Python использует открытую адресацию с псевдослучайным probing.


## Сложность операций

| Операция | Avg | Worst |
|---|---|---|
| `d[k]` (get) | O(1) | O(n) |
| `d[k] = v` (set) | O(1) | O(n) |
| `del d[k]` | O(1) | O(n) |
| `k in d` | O(1) | O(n) |

Worst case — если у всех ключей одинаковый хеш (специально подобранные ключи / атака на хеш).

## Что может быть ключом

Только **хешируемые** (`__hash__`) и **неизменяемые** объекты:
- ✅ `int`, `float`, `str`, `bytes`, `tuple` (из хешируемых), `frozenset`
- ❌ `list`, `dict`, `set` (изменяемые)


In [ ]:
d = {(1, 2): 'point'}        # tuple — ок
print(d[(1, 2)])

try:
    d[[1, 2]] = 'fail'       # list — TypeError
except TypeError as e:
    print('TypeError:', e)

## Типичные задачи на хеш-таблицу


### Найти дубликаты в массиве


In [ ]:
def has_duplicates(arr):
    seen = set()
    for x in arr:
        if x in seen:
            return True
        seen.add(x)
    return False

has_duplicates([1, 2, 3, 4, 2])   # True

### Two Sum — пара чисел с заданной суммой за O(n)


In [ ]:
def two_sum(arr, target):
    seen = {}                # value → index
    for i, x in enumerate(arr):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return None

two_sum([2, 7, 11, 15], 9)   # [0, 1]

### Подсчёт частот


In [ ]:
from collections import Counter

Counter('mississippi')
# Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})

### Группировка анаграмм


In [ ]:
from collections import defaultdict

def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        key = ''.join(sorted(w))   # отсортированные буквы — общий ключ
        groups[key].append(w)
    return list(groups.values())

group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat'])

## Реализация простой хеш-таблицы с методом цепочек


In [ ]:
class HashMap:
    def __init__(self, capacity=16):
        self.capacity = capacity
        self.buckets = [[] for _ in range(capacity)]
        self.size = 0

    def _index(self, key):
        return hash(key) % self.capacity

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self.size += 1

    def get(self, key):
        for k, v in self.buckets[self._index(key)]:
            if k == key:
                return v
        raise KeyError(key)

    def remove(self, key):
        bucket = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return
        raise KeyError(key)

m = HashMap()
m.put('a', 1); m.put('b', 2); m.put('a', 10)
print(m.get('a'), m.get('b'), m.size)   # 10 2 2

## Что спрашивают на собеседовании

- В чём разница между `dict` и `set`?
- Почему `list` нельзя использовать как ключ? *(изменяемый, нет `__hash__`.)*
- Что такое **коллизия** и как с ней борются?
- Какие гарантии порядка у `dict`? *(С Python 3.7 — порядок вставки гарантирован.)*
- Когда `O(1)` превращается в `O(n)`? *(При плохой хеш-функции и большом числе коллизий.)*
